In [1]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers faiss-cpu langchain langchain-community
!pip install -q requests newspaper3k lxml_html_clean
print('✅ All packages installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━

In [2]:
# Cell 2 — Load API keys from Colab Secrets
from google.colab import userdata

ADZUNA_APP_ID  = userdata.get('ADZUNA_APP_ID')
ADZUNA_APP_KEY = userdata.get('ADZUNA_APP_KEY')

print('✅ API keys loaded!')
print(f'App ID: {ADZUNA_APP_ID[:4]}****')
print(f'App Key: {ADZUNA_APP_KEY[:4]}****')

✅ API keys loaded!
App ID: 53ee****
App Key: 1c0e****


In [3]:
# Cell 3 — Fetch job data from Adzuna API
import requests
import json

def fetch_adzuna_jobs(query, location='canada', pages=5):
    all_jobs = []
    base_url = 'https://api.adzuna.com/v1/api/jobs/ca/search'

    for page in range(1, pages + 1):
        params = {
            'app_id': ADZUNA_APP_ID,
            'app_key': ADZUNA_APP_KEY,
            'results_per_page': 20,
            'what': query,
            'where': location,
            'content-type': 'application/json'
        }
        response = requests.get(f'{base_url}/{page}', params=params)
        if response.status_code == 200:
            data = response.json()
            jobs = data.get('results', [])
            all_jobs.extend(jobs)
            print(f'Page {page}: {len(jobs)} jobs fetched')
        else:
            print(f'Page {page} error: {response.status_code}')

    return all_jobs

# Fetch different job categories
print('Fetching AI/ML jobs...')
ai_jobs = fetch_adzuna_jobs('artificial intelligence machine learning')

print('\nFetching tech layoff/software jobs...')
tech_jobs = fetch_adzuna_jobs('software engineer developer')

print('\nFetching data science jobs...')
data_jobs = fetch_adzuna_jobs('data scientist analyst')

all_jobs = ai_jobs + tech_jobs + data_jobs
print(f'\n✅ Total jobs fetched: {len(all_jobs)}')

Fetching AI/ML jobs...
Page 1: 20 jobs fetched
Page 2: 20 jobs fetched
Page 3: 20 jobs fetched
Page 4: 20 jobs fetched
Page 5: 20 jobs fetched

Fetching tech layoff/software jobs...
Page 1: 20 jobs fetched
Page 2: 20 jobs fetched
Page 3: 20 jobs fetched
Page 4: 20 jobs fetched
Page 5: 20 jobs fetched

Fetching data science jobs...
Page 1: 20 jobs fetched
Page 2: 20 jobs fetched
Page 3: 20 jobs fetched
Page 4: 20 jobs fetched
Page 5: 20 jobs fetched

✅ Total jobs fetched: 300


In [4]:
# Cell 4 — Convert jobs to text chunks for RAG
def job_to_text(job):
    title    = job.get('title', 'Unknown Title')
    company  = job.get('company', {}).get('display_name', 'Unknown Company')
    location = job.get('location', {}).get('display_name', 'Canada')
    desc     = job.get('description', '')[:500]
    salary   = job.get('salary_min', 'Not specified')
    category = job.get('category', {}).get('label', 'Technology')

    return (
        f"Job Title: {title}\n"
        f"Company: {company}\n"
        f"Location: {location}\n"
        f"Category: {category}\n"
        f"Salary: {salary}\n"
        f"Description: {desc}\n"
        f"---"
    )

# Convert all jobs to text
chunks = [job_to_text(job) for job in all_jobs]

# Remove empty chunks
chunks = [c for c in chunks if len(c) > 50]

print(f'✅ Total chunks created: {len(chunks)}')
print('\nSample chunk:')
print(chunks[0])

✅ Total chunks created: 300

Sample chunk:
Job Title: Process and Tools - Artificial Intelligence / Machine Learning
Company: Bombardier
Location: Dorval, Montréal
Category: IT Jobs
Salary: Not specified
Description: _When applicable, Bombardier promotes flexible and hybrid work policies._ Why join us? At Bombardier, we design, build and maintain the world's peak-performing aircraft for the world's most discerning people and businesses, governments and militaries. We have been successful in setting the highest standards by putting our people at the heart of it all, and defining excellence, together. Working at Bombardier means operating at the highest level. Every day, you are part of a team that delivers s…
---


In [5]:
# Cell 5 — Create embeddings and FAISS index
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('Creating embeddings for 300 chunks...')
embeddings = embedder.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f'\nEmbedding shape: {embeddings.shape}')

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f'✅ FAISS index built!')
print(f'Total vectors in index: {index.ntotal}')

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings for 300 chunks...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Embedding shape: (300, 384)
✅ FAISS index built!
Total vectors in index: 300


In [6]:
# Cell 6 — Test RAG search
def search_jobs(query, top_k=3):
    query_embedding = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    print(f'Top {top_k} results for: "{query}"\n')
    for i, idx in enumerate(indices[0]):
        print(f'Result {i+1}:')
        print(chunks[idx][:300])
        print()

# Test queries
search_jobs('machine learning engineer skills Canada')
print('=' * 50)
search_jobs('data scientist salary Toronto')
print('=' * 50)
search_jobs('software engineer layoffs tech jobs')

Top 3 results for: "machine learning engineer skills Canada"

Result 1:
Job Title: Senior Machine Learning Engineer - Artificial Intelligence
Company: Bloomberg
Location: Toronto, Ontario
Category: IT Jobs
Salary: Not specified
Description: Senior Machine Learning Engineer - Artificial Intelligence Location Toronto Business Area Engineering and CTO Ref  10044510 Descrip

Result 2:
Job Title: Software Engineer, Backend
Company: Lyft
Location: Toronto, Ontario
Category: IT Jobs
Salary: 108000
Description: At Lyft, our purpose is to serve and connect. We aim to achieve this by cultivating a work environment where all team members belong and have the opportunity to thrive. With o

Result 3:
Job Title: Process and Tools - Artificial Intelligence / Machine Learning
Company: Bombardier
Location: Mississauga, Peel region
Category: IT Jobs
Salary: Not specified
Description: _When applicable, Bombardier promotes flexible and hybrid work policies._ Why join us? At Bombardier, we design, build

To

In [7]:
# Cell 7 — Save FAISS index and chunks to Google Drive
from google.colab import drive
import pickle
import faiss
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create folder
save_dir = '/content/drive/MyDrive/job_market_navigator'
os.makedirs(save_dir, exist_ok=True)

# Save FAISS index
faiss.write_index(index, f'{save_dir}/faiss_index.bin')

# Save chunks (text data)
with open(f'{save_dir}/chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

print(f'✅ FAISS index saved!')
print(f'✅ Chunks saved!')
print(f'📁 Location: {save_dir}')
print(f'Total chunks: {len(chunks)}')

Mounted at /content/drive
✅ FAISS index saved!
✅ Chunks saved!
📁 Location: /content/drive/MyDrive/job_market_navigator
Total chunks: 300


In [8]:
# Cell 8 — Verify saved files + final test
import os
import pickle
import faiss

save_dir = '/content/drive/MyDrive/job_market_navigator'

# Check files exist
files = os.listdir(save_dir)
print('Files in Drive folder:')
for f in files:
    size = os.path.getsize(f'{save_dir}/{f}') / 1024
    print(f'  {f} — {size:.1f} KB')

# Reload and verify
print('\nReloading index from Drive...')
loaded_index = faiss.read_index(f'{save_dir}/faiss_index.bin')
with open(f'{save_dir}/chunks.pkl', 'rb') as f:
    loaded_chunks = pickle.load(f)

print(f'✅ Index reloaded — vectors: {loaded_index.ntotal}')
print(f'✅ Chunks reloaded — total: {len(loaded_chunks)}')
print('\n✅ Notebook 2 Complete! Ready for Notebook 3 — Agentic System!')

Files in Drive folder:
  faiss_index.bin — 450.0 KB
  chunks.pkl — 195.5 KB

Reloading index from Drive...
✅ Index reloaded — vectors: 300
✅ Chunks reloaded — total: 300

✅ Notebook 2 Complete! Ready for Notebook 3 — Agentic System!


In [9]:
# Notebook 2 mein new cell add karo — end mein
from huggingface_hub import HfApi

api = HfApi()

print('Uploading faiss_index.bin...')
api.upload_file(
    path_or_fileobj='/content/drive/MyDrive/job_market_navigator/faiss_index.bin',
    path_in_repo='faiss_index.bin',
    repo_id='kashanikram/job-market-navigator-distilgpt2',
    repo_type='model'
)
print('✅ faiss_index.bin uploaded!')

print('Uploading chunks.pkl...')
api.upload_file(
    path_or_fileobj='/content/drive/MyDrive/job_market_navigator/chunks.pkl',
    path_in_repo='chunks.pkl',
    repo_id='kashanikram/job-market-navigator-distilgpt2',
    repo_type='model'
)
print('✅ chunks.pkl uploaded!')
print('\n✅ Both files on HuggingFace Hub!')

Uploading faiss_index.bin...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...navigator/faiss_index.bin: 100%|##########|  461kB /  461kB            

  ...navigator/faiss_index.bin: 100%|##########|  461kB /  461kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...rket_navigator/chunks.pkl: 100%|##########|  200kB /  200kB            

  ...rket_navigator/chunks.pkl: 100%|##########|  200kB /  200kB            

✅ chunks.pkl uploaded!

✅ Both files on HuggingFace Hub!
